In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

SEED = 777

In [3]:
# Carga
basic = pd.read_csv('../data/scaled/data_basic.csv')

print(basic.shape)

print(basic.dtypes.value_counts())

print(basic.head())

# Nulos
nulos = basic.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin nulos")

# Duplicados
print(f"{basic.duplicated().sum()} filas duplicadas")

# Clases
print(basic['Variable de Salida'].value_counts())
print(basic['Variable de Salida'].value_counts(normalize=True).round(3))


(9995, 132)
float64    67
bool       36
int64      29
Name: count, dtype: int64
   Variable de Salida  Variable 03  Variable 04  Variable 05  Variable 06  \
0                   1          3.0          6.0         13.0         7.95   
1                   1          2.0          6.0         13.0         5.50   
2                   1          3.0          6.0         13.0         6.00   
3                   0          3.0          6.0         13.0         5.50   
4                   1          3.0          6.0         13.0         6.00   

   Variable 07  Variable 08  Variable 09  Variable 10  Variable 11  ...  \
0    17.123810    24.800001        169.3   169.570004    169.90001  ...   
1    17.166667    24.700000        169.3   169.510002    169.90001  ...   
2    17.126087    24.800001        169.2   169.560006    169.90001  ...   
3    17.152174    24.800001        169.3   169.344447    169.50000  ...   
4    17.145455    24.800001        169.2   169.640007    169.90001  ...   

   Var

In [8]:
# ── 1. IDENTIFICAR TODAS LAS VARIABLES CATEGÓRICAS REALES ────────────────────
print("=== VARIABLES BINARIAS DISFRAZADAS DE INT (corregido) ===")
binary_int = ['Variable 24', 'Variable 25', 'Variable 26', 'Variable 36', 
              'Variable 37', 'Variable 38', 'Variable 48', 'Variable 49', 'Variable 50']

for c in binary_int:
    vals = sorted(basic[c].unique())
    v0, v1 = vals[0], vals[1]
    nok_v0 = basic.loc[basic[c]==v0, 'Variable de Salida'].mean()
    nok_v1 = basic.loc[basic[c]==v1, 'Variable de Salida'].mean()
    n0 = (basic[c]==v0).sum()
    n1 = (basic[c]==v1).sum()
    print(f"  {c}: [{v0}]={n0} muestras NOK:{nok_v0:.1%} | [{v1}]={n1} muestras NOK:{nok_v1:.1%}")


=== VARIABLES BINARIAS DISFRAZADAS DE INT (corregido) ===
  Variable 24: [152]=4765 muestras NOK:75.3% | [155]=5230 muestras NOK:75.9%
  Variable 25: [152]=4765 muestras NOK:75.3% | [155]=5230 muestras NOK:75.9%
  Variable 26: [152]=4765 muestras NOK:75.3% | [155]=5230 muestras NOK:75.9%
  Variable 36: [152]=4765 muestras NOK:75.3% | [155]=5230 muestras NOK:75.9%
  Variable 37: [152]=4765 muestras NOK:75.3% | [155]=5230 muestras NOK:75.9%
  Variable 38: [152]=4765 muestras NOK:75.3% | [155]=5230 muestras NOK:75.9%
  Variable 48: [134]=5230 muestras NOK:75.9% | [135]=4765 muestras NOK:75.3%
  Variable 49: [134]=5230 muestras NOK:75.9% | [135]=4765 muestras NOK:75.3%
  Variable 50: [134]=5230 muestras NOK:75.9% | [135]=4765 muestras NOK:75.3%


Las Variables 24-50 (binarias) son completamente inútiles. Ambos valores tienen exactamente el mismo NOK rate (75.3% vs 75.9% — prácticamente idéntico al rate global de 75.6%). No aportan ninguna señal discriminativa. Los patrones de split son idénticos entre grupos:

Variables 24, 25, 26, 36, 37, 38 → exactamente el mismo split 4765/5230
Variables 48, 49, 50 → split invertido 5230/4765 (que es el mismo split)

Esto huele a que estas 9 variables son en realidad la misma variable subyacente codificada de formas distintas, o derivan de la misma fuente. Son redundantes entre sí y además no discriminan. Candidatas directas a eliminación.

In [6]:
# ── 2. IDENTIFICAR GRUPOS DE VARIABLES (por número en nombre) ─────────────────
print("\n=== GRUPOS DE VARIABLES ONE-HOT ===")
bool_cols = basic.select_dtypes(include='bool').columns.tolist()
prefixes = pd.Series([c.rsplit('_', 1)[0] for c in bool_cols]).value_counts()
print(prefixes)

print("\n=== TASA DE NOK POR CATEGORÍA — TODAS LAS VARS ONE-HOT ===")
for prefix in prefixes.index:
    cols = [c for c in bool_cols if c.startswith(prefix + '_')]
    print(f"\n{prefix} ({len(cols)} categorías):")
    for col in cols:
        mask = basic[col] == True
        if mask.sum() >= 5:
            nok_rate = basic.loc[mask, 'Variable de Salida'].mean()
            n = mask.sum()
            print(f"  {col.replace(prefix+'_',''):>8}: {n:>5} muestras — NOK rate: {nok_rate:.1%}")
        else:
            print(f"  {col.replace(prefix+'_',''):>8}: {mask.sum():>5} muestras — muy pocas, considerar agrupar")




=== GRUPOS DE VARIABLES ONE-HOT ===
Variable 96     15
Variable 101    11
Variable 92      4
Variable 91      4
Variable 01      2
Name: count, dtype: int64

=== TASA DE NOK POR CATEGORÍA — TODAS LAS VARS ONE-HOT ===

Variable 96 (15 categorías):
        AM:     2 muestras — muy pocas, considerar agrupar
        AO:    35 muestras — NOK rate: 68.6%
        AR:    81 muestras — NOK rate: 76.5%
        AZ:    57 muestras — NOK rate: 73.7%
        BO:    32 muestras — NOK rate: 65.6%
        BR:  8342 muestras — NOK rate: 75.5%
        CA:     1 muestras — muy pocas, considerar agrupar
        CI:   182 muestras — NOK rate: 69.2%
        GB:     1 muestras — muy pocas, considerar agrupar
        HU:   124 muestras — NOK rate: 73.4%
        PR:  1056 muestras — NOK rate: 78.4%
        RE:     4 muestras — muy pocas, considerar agrupar
        VD:    12 muestras — NOK rate: 58.3%
        VE:    64 muestras — NOK rate: 78.1%
        VR:     2 muestras — muy pocas, considerar agrupar

Variab

Variable 96 es casi idéntica a Variable 101 — mismas categorías (AR, AZ, BR, PR, VE, VD...), distribuciones muy similares, y NOK rates parecidos. Probablemente son la misma variable de origen o una derivada. Hay que cruzarlas para confirmarlo.
Variables 91 y 92 son sospechosamente similares entre sí — mismas 4 categorías (A, B, FDS, N), distribuciones casi idénticas, NOK rates prácticamente iguales. Probablemente miden lo mismo en momentos distintos o son redundantes.
Variable 01 es el mismo split que las binarias 24-50 — exactamente 4765 DERECHA y 5230 IZQUIERDA, con NOK rates 75.3%/75.9%. Es la misma variable subyacente que está codificada en 9 variables enteras más. Todo ese bloque es una sola variable real.

In [10]:
# ¿Variable 01 es idéntica a las binarias int?
v01_num = basic['Variable 01_IZQUIERDA'].astype(int)

binary_int = ['Variable 24', 'Variable 25', 'Variable 26', 'Variable 36', 
              'Variable 37', 'Variable 38', 'Variable 48', 'Variable 49', 'Variable 50']

print("=== ¿Variable 01 == Variables binarias int? ===")
for col in binary_int:
    vals = sorted(basic[col].unique())
    v_norm = (basic[col] == vals[1]).astype(int)
    match = (v01_num == v_norm).mean()
    print(f"  Variable 01 == {col}: {match:.1%} coincidencia")

=== ¿Variable 01 == Variables binarias int? ===
  Variable 01 == Variable 24: 100.0% coincidencia
  Variable 01 == Variable 25: 100.0% coincidencia
  Variable 01 == Variable 26: 100.0% coincidencia
  Variable 01 == Variable 36: 100.0% coincidencia
  Variable 01 == Variable 37: 100.0% coincidencia
  Variable 01 == Variable 38: 100.0% coincidencia
  Variable 01 == Variable 48: 0.0% coincidencia
  Variable 01 == Variable 49: 0.0% coincidencia
  Variable 01 == Variable 50: 0.0% coincidencia


Son exactamente iguales

In [11]:
# ¿Variable 96 y Variable 101 son la misma?
bool_cols = basic.select_dtypes(include='bool').columns.tolist()

cols_96  = [c for c in bool_cols if c.startswith('Variable 96_')]
cols_101 = [c for c in bool_cols if c.startswith('Variable 101_')]

cats_96  = set(c.replace('Variable 96_', '')  for c in cols_96)
cats_101 = set(c.replace('Variable 101_', '') for c in cols_101)

print(f"Categorías solo en 96:  {sorted(cats_96 - cats_101)}")
print(f"Categorías solo en 101: {sorted(cats_101 - cats_96)}")
print(f"Categorías en común:    {sorted(cats_96 & cats_101)}")

print("\n¿Coinciden fila a fila en categorías comunes?")
for cat in sorted(cats_96 & cats_101):
    match = (basic[f'Variable 96_{cat}'] == basic[f'Variable 101_{cat}']).mean()
    print(f"  {cat}: {match:.1%} coincidencia")

Categorías solo en 96:  ['AO', 'BO', 'CI', 'HU', 'RE']
Categorías solo en 101: ['VB']
Categorías en común:    ['AM', 'AR', 'AZ', 'BR', 'CA', 'GB', 'PR', 'VD', 'VE', 'VR']

¿Coinciden fila a fila en categorías comunes?
  AM: 99.9% coincidencia
  AR: 98.4% coincidencia
  AZ: 98.7% coincidencia
  BR: 83.8% coincidencia
  CA: 100.0% coincidencia
  GB: 100.0% coincidencia
  PR: 87.7% coincidencia
  VD: 99.8% coincidencia
  VE: 98.7% coincidencia
  VR: 100.0% coincidencia


Parecidas pero distintas, hay que conservar ambas.


In [12]:
# ¿Variable 91 y Variable 92 son la misma?
cols_91 = [c for c in bool_cols if c.startswith('Variable 91_')]
cols_92 = [c for c in bool_cols if c.startswith('Variable 92_')]

print("¿Coinciden fila a fila?")
for cat in ['A', 'B', 'FDS', 'N']:
    col_91 = f'Variable 91_{cat}'
    col_92 = f'Variable 92_{cat}'
    match = (basic[col_91] == basic[col_92]).mean()
    print(f"  {cat}: {match:.1%} coincidencia")

¿Coinciden fila a fila?
  A: 96.5% coincidencia
  B: 96.9% coincidencia
  FDS: 99.8% coincidencia
  N: 96.6% coincidencia


Igual que anterior

In [13]:
print("=== OUTLIERS (valores > 5 std de la media) ===")
float_cols = basic.select_dtypes(include='float64').columns.tolist()
z_scores = ((basic[float_cols] - basic[float_cols].mean()) / basic[float_cols].std()).abs()
outliers_per_feature = (z_scores > 5).sum().sort_values(ascending=False)
print(outliers_per_feature[outliers_per_feature > 0].head(15))
print(f"\nFilas con al menos 1 outlier extremo: {(z_scores > 5).any(axis=1).sum()}")
print(f"Filas con más de 5 outliers extremos: {(z_scores > 5).sum(axis=1).gt(5).sum()}")

=== OUTLIERS (valores > 5 std de la media) ===
Variable 99    103
Variable 03     83
Variable 80     80
Variable 72     78
Variable 57     75
Variable 60     73
Variable 05     64
Variable 79     62
Variable 73     62
Variable 78     62
Variable 74     62
Variable 75     62
Variable 77     62
Variable 64     56
Variable 06     56
dtype: int64

Filas con al menos 1 outlier extremo: 765
Filas con más de 5 outliers extremos: 67


765 filas con outliers extremos es el 7.6% del dataset — no es despreciable. Y los 67 con más de 5 outliers simultáneos son sospechosos, podrían ser mediciones corruptas o condiciones de operación muy atípicas.

In [14]:
outlier_mask = (z_scores > 5).any(axis=1)
outlier_extremo_mask = (z_scores > 5).sum(axis=1).gt(5)

print("=== ¿LOS OUTLIERS SE CONCENTRAN EN ALGUNA CLASE? ===")
print(f"\nFilas con algún outlier extremo (>5std):")
print(basic.loc[outlier_mask, 'Variable de Salida'].value_counts(normalize=True).round(3))
print(f"(baseline global: NOK=0.756, OK=0.244)")

print(f"\nFilas con más de 5 outliers simultáneos:")
print(basic.loc[outlier_extremo_mask, 'Variable de Salida'].value_counts(normalize=True).round(3))

=== ¿LOS OUTLIERS SE CONCENTRAN EN ALGUNA CLASE? ===

Filas con algún outlier extremo (>5std):
Variable de Salida
1    0.771
0    0.229
Name: proportion, dtype: float64
(baseline global: NOK=0.756, OK=0.244)

Filas con más de 5 outliers simultáneos:
Variable de Salida
1    0.836
0    0.164
Name: proportion, dtype: float64


Los outliers siguen la distribución global (77% vs 75.6% baseline) — no hay concentración significativa en ninguna clase. No son errores de medición sistemáticos de una clase concreta, simplemente variabilidad natural del proceso.
Los 67 con más de 5 outliers simultáneos sí tienen algo más de NOK (83.6% vs 75.6%), pero son tan pocos que no justifica eliminarlos. Los dejamos.

In [15]:
print("=== CORRELACIONES ALTAS ENTRE FEATURES (float) ===")
float_cols = basic.select_dtypes(include='float64').columns.tolist()
corr_matrix = basic[float_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = [
    (c, r, upper.loc[r, c])
    for c in upper.columns
    for r in upper.index
    if upper.loc[r, c] > 0.8
]
high_corr_pairs.sort(key=lambda x: -x[2])

print(f"Pares con correlación > 0.8: {len(high_corr_pairs)}")
for a, b, v in high_corr_pairs:
    print(f"  {a} ↔ {b}: {v:.3f}")

=== CORRELACIONES ALTAS ENTRE FEATURES (float) ===
Pares con correlación > 0.8: 98
  Variable 78 ↔ Variable 77: 0.999
  Variable 86 ↔ Variable 07: 0.999
  Variable 66 ↔ Variable 65: 0.998
  Variable 86 ↔ Variable 85: 0.998
  Variable 52 ↔ Variable 51: 0.998
  Variable 53 ↔ Variable 52: 0.998
  Variable 79 ↔ Variable 78: 0.998
  Variable 67 ↔ Variable 66: 0.998
  Variable 87 ↔ Variable 86: 0.997
  Variable 85 ↔ Variable 07: 0.997
  Variable 17 ↔ Variable 16: 0.996
  Variable 16 ↔ Variable 15: 0.996
  Variable 74 ↔ Variable 73: 0.996
  Variable 87 ↔ Variable 07: 0.995
  Variable 79 ↔ Variable 77: 0.995
  Variable 67 ↔ Variable 65: 0.994
  Variable 86 ↔ Variable 66: 0.994
  Variable 87 ↔ Variable 85: 0.993
  Variable 66 ↔ Variable 07: 0.993
  Variable 85 ↔ Variable 66: 0.993
  Variable 19 ↔ Variable 18: 0.993
  Variable 53 ↔ Variable 51: 0.993
  Variable 75 ↔ Variable 74: 0.992
  Variable 86 ↔ Variable 65: 0.992
  Variable 85 ↔ Variable 65: 0.991
  Variable 65 ↔ Variable 07: 0.991
  Varia

98 pares con correlación > 0.8 es muchísimo. Hay grupos de variables que son prácticamente lo mismo medido varias veces — Variable 07, 65, 66, 67, 77, 78, 79, 85, 86, 87 forman un cluster casi perfecto entre sí.

In [16]:
# Construir grupos usando union-find simple
threshold = 0.8
features = float_cols
corr_matrix = basic[features].corr().abs()

# Asignar cada feature a un grupo
grupos = {f: f for f in features}  # cada feature es su propio representante

def find(x):
    while grupos[x] != x:
        grupos[x] = grupos[grupos[x]]
        x = grupos[x]
    return x

def union(x, y):
    grupos[find(x)] = find(y)

for i, col_a in enumerate(features):
    for col_b in features[i+1:]:
        if corr_matrix.loc[col_a, col_b] > threshold:
            union(col_a, col_b)

# Agrupar
from collections import defaultdict
clusters = defaultdict(list)
for f in features:
    clusters[find(f)].append(f)

clusters_multi = {k: v for k, v in clusters.items() if len(v) > 1}
singletons = [k for k, v in clusters.items() if len(v) == 1]

print(f"=== CLUSTERS DE FEATURES CORRELADAS (>{threshold}) ===")
print(f"Grupos con más de 1 variable: {len(clusters_multi)}")
print(f"Variables únicas (sin correlación alta): {len(singletons)}")
print(f"\nTotal variables float originales: {len(features)}")
print(f"Variables independientes efectivas: {len(clusters_multi) + len(singletons)}")

print(f"\n=== DETALLE DE GRUPOS ===")
for rep, members in sorted(clusters_multi.items(), key=lambda x: -len(x[1])):
    print(f"\nGrupo ({len(members)} variables): {members}")

=== CLUSTERS DE FEATURES CORRELADAS (>0.8) ===
Grupos con más de 1 variable: 9
Variables únicas (sin correlación alta): 23

Total variables float originales: 67
Variables independientes efectivas: 32

=== DETALLE DE GRUPOS ===

Grupo (12 variables): ['Variable 15', 'Variable 16', 'Variable 17', 'Variable 18', 'Variable 19', 'Variable 20', 'Variable 27', 'Variable 28', 'Variable 29', 'Variable 30', 'Variable 31', 'Variable 32']

Grupo (10 variables): ['Variable 07', 'Variable 65', 'Variable 66', 'Variable 67', 'Variable 77', 'Variable 78', 'Variable 79', 'Variable 85', 'Variable 86', 'Variable 87']

Grupo (4 variables): ['Variable 39', 'Variable 40', 'Variable 41', 'Variable 47']

Grupo (3 variables): ['Variable 09', 'Variable 10', 'Variable 11']

Grupo (3 variables): ['Variable 42', 'Variable 43', 'Variable 44']

Grupo (3 variables): ['Variable 51', 'Variable 52', 'Variable 53']

Grupo (3 variables): ['Variable 61', 'Variable 62', 'Variable 63']

Grupo (3 variables): ['Variable 73', 'V

De 67 variables float, en realidad solo hay 32 señales independientes — los 9 grupos reducen 44 variables a 9 representantes.

In [17]:
from scipy.stats import mannwhitneyu

print("=== MEJOR VARIABLE DE CADA GRUPO (por Mann-Whitney) ===")

grupos_detalle = {
    'G1_12vars': ['Variable 15', 'Variable 16', 'Variable 17', 'Variable 18', 'Variable 19', 'Variable 20', 'Variable 27', 'Variable 28', 'Variable 29', 'Variable 30', 'Variable 31', 'Variable 32'],
    'G2_10vars': ['Variable 07', 'Variable 65', 'Variable 66', 'Variable 67', 'Variable 77', 'Variable 78', 'Variable 79', 'Variable 85', 'Variable 86', 'Variable 87'],
    'G3_4vars':  ['Variable 39', 'Variable 40', 'Variable 41', 'Variable 47'],
    'G4_3vars':  ['Variable 09', 'Variable 10', 'Variable 11'],
    'G5_3vars':  ['Variable 42', 'Variable 43', 'Variable 44'],
    'G6_3vars':  ['Variable 51', 'Variable 52', 'Variable 53'],
    'G7_3vars':  ['Variable 61', 'Variable 62', 'Variable 63'],
    'G8_3vars':  ['Variable 73', 'Variable 74', 'Variable 75'],
    'G9_3vars':  ['Variable 81', 'Variable 82', 'Variable 83'],
}

ok_mask  = basic['Variable de Salida'] == 0
nok_mask = basic['Variable de Salida'] == 1

mejores = {}
for grupo, vars_ in grupos_detalle.items():
    resultados = []
    for col in vars_:
        stat, p = mannwhitneyu(basic.loc[ok_mask, col], basic.loc[nok_mask, col], alternative='two-sided')
        resultados.append((col, p))
    resultados.sort(key=lambda x: x[1])
    mejor = resultados[0]
    peor  = resultados[-1]
    mejores[grupo] = mejor[0]
    print(f"\n{grupo}:")
    print(f"  Mejor:  {mejor[0]} (p={mejor[1]:.2e})")
    print(f"  Peor:   {peor[0]}  (p={peor[1]:.2e})")
    print(f"  Todas:  {[f'{v}(p={p:.0e})' for v,p in resultados]}")

print(f"\n=== REPRESENTANTES SELECCIONADOS ===")
for g, v in mejores.items():
    print(f"  {g}: {v}")

=== MEJOR VARIABLE DE CADA GRUPO (por Mann-Whitney) ===

G1_12vars:
  Mejor:  Variable 29 (p=9.81e-12)
  Peor:   Variable 30  (p=1.28e-06)
  Todas:  ['Variable 29(p=1e-11)', 'Variable 20(p=4e-11)', 'Variable 28(p=4e-10)', 'Variable 19(p=7e-10)', 'Variable 17(p=2e-09)', 'Variable 27(p=4e-09)', 'Variable 15(p=7e-09)', 'Variable 16(p=7e-09)', 'Variable 18(p=1e-08)', 'Variable 32(p=1e-07)', 'Variable 31(p=6e-07)', 'Variable 30(p=1e-06)']

G2_10vars:
  Mejor:  Variable 07 (p=1.10e-04)
  Peor:   Variable 66  (p=3.79e-04)
  Todas:  ['Variable 07(p=1e-04)', 'Variable 87(p=1e-04)', 'Variable 86(p=2e-04)', 'Variable 85(p=2e-04)', 'Variable 77(p=2e-04)', 'Variable 65(p=3e-04)', 'Variable 79(p=3e-04)', 'Variable 67(p=3e-04)', 'Variable 78(p=3e-04)', 'Variable 66(p=4e-04)']

G3_4vars:
  Mejor:  Variable 47 (p=3.38e-02)
  Peor:   Variable 39  (p=1.71e-01)
  Todas:  ['Variable 47(p=3e-02)', 'Variable 41(p=6e-02)', 'Variable 40(p=1e-01)', 'Variable 39(p=2e-01)']

G4_3vars:
  Mejor:  Variable 10 (p=9.1

Hay algo importante aquí antes de tomar decisiones. El grupo G3 es preocupante — la mejor variable (Variable 47) tiene p=0.03, que es apenas significativa, y Variable 39 directamente no discrimina (p=0.17). Este grupo entero es señal débil.
Pero antes de decidir si nos quedamos con el representante o con el grupo completo, necesitamos el Mann-Whitney para todas las variables del dataset, no solo los grupos. Así tenemos el mapa completo de qué aporta y qué no:

In [18]:
from scipy.stats import mannwhitneyu

ok_mask  = basic['Variable de Salida'] == 0
nok_mask = basic['Variable de Salida'] == 1

# Todas las columnas excepto el target
all_cols = [c for c in basic.columns if c != 'Variable de Salida']

resultados = []
for col in all_cols:
    vals_ok  = basic.loc[ok_mask,  col].astype(float)
    vals_nok = basic.loc[nok_mask, col].astype(float)
    stat, p  = mannwhitneyu(vals_ok, vals_nok, alternative='two-sided')
    resultados.append({'feature': col, 'p_value': p})

mw_df = pd.DataFrame(resultados).sort_values('p_value').reset_index(drop=True)

print(f"=== RESUMEN DISCRIMINACIÓN (Mann-Whitney) ===")
print(f"Total features:              {len(mw_df)}")
print(f"p < 0.001 (muy significativas): {(mw_df['p_value'] < 0.001).sum()}")
print(f"p < 0.05  (significativas):     {(mw_df['p_value'] < 0.05).sum()}")
print(f"p >= 0.05 (no discriminan):     {(mw_df['p_value'] >= 0.05).sum()}")

print(f"\n=== TOP 25 MÁS DISCRIMINATIVAS ===")
print(mw_df.head(25).to_string(index=False))

print(f"\n=== NO DISCRIMINAN (p >= 0.05) ===")
print(mw_df[mw_df['p_value'] >= 0.05]['feature'].tolist())

=== RESUMEN DISCRIMINACIÓN (Mann-Whitney) ===
Total features:              131
p < 0.001 (muy significativas): 39
p < 0.05  (significativas):     68
p >= 0.05 (no discriminan):     63

=== TOP 25 MÁS DISCRIMINATIVAS ===
        feature      p_value
Variable 101_PR 6.012166e-30
Variable 101_BR 7.561007e-24
    Variable 29 9.808885e-12
    Variable 20 3.640251e-11
    Variable 28 4.089519e-10
    Variable 19 6.835851e-10
    Variable 17 1.900940e-09
    Variable 27 4.029953e-09
    Variable 15 6.983947e-09
    Variable 16 7.319632e-09
    Variable 18 1.035524e-08
    Variable 63 3.741776e-08
    Variable 32 1.223675e-07
    Variable 62 2.046931e-07
    Variable 61 4.078974e-07
    Variable 31 6.031331e-07
    Variable 74 1.083239e-06
    Variable 73 1.095503e-06
    Variable 30 1.277560e-06
    Variable 75 1.915862e-06
Variable 101_AR 2.244580e-06
    Variable 99 5.342831e-05
    Variable 07 1.101356e-04
    Variable 87 1.117183e-04
    Variable 86 1.586464e-04

=== NO DISCRIMINAN (p >= 

De 131 features, 63 no discriminan en absoluto — y además confirmamos lo que sospechábamos: todo el bloque Variable 01 + Variables 24-50 está ahí, y muchas dummies de Variable 96 que son categorías raras con pocas muestras.
Hagamos el resumen final de la exploración antes de pasar a construir el dataset limpio:

In [19]:
print("=== RESUMEN FINAL DE EXPLORACIÓN ===")

# Bloque 1: redundantes confirmadas
redundantes = ['Variable 24', 'Variable 25', 'Variable 26', 
               'Variable 36', 'Variable 37', 'Variable 38',
               'Variable 48', 'Variable 49', 'Variable 50']
print(f"\n1. REDUNDANTES EXACTAS (copias de Variable 01): {len(redundantes)}")
print(f"   {redundantes}")

# Bloque 2: no discriminan
no_discriminan = mw_df[mw_df['p_value'] >= 0.05]['feature'].tolist()
# quitar las que ya están en redundantes
no_discriminan_extra = [f for f in no_discriminan if f not in redundantes]
print(f"\n2. NO DISCRIMINAN (p>=0.05, sin contar redundantes): {len(no_discriminan_extra)}")

# Bloque 3: discriminan
discriminan = mw_df[mw_df['p_value'] < 0.05]['feature'].tolist()
print(f"\n3. DISCRIMINAN (p<0.05): {len(discriminan)}")
print(f"   Muy significativas (p<0.001): {(mw_df['p_value'] < 0.001).sum()}")

# Bloque 4: grupos correlados
print(f"\n4. GRUPOS DE FEATURES CORRELADAS (>0.8):")
print(f"   9 grupos → 44 variables colapsables a 9 representantes")
print(f"   Representantes: Variable 29, 07, 47, 10, 42, 53, 63, 74, 82")

print(f"\n=== DECISIONES PARA EL DATASET LIMPIO ===")
print(f"""
   ELIMINAR:
   - 9 variables redundantes exactas (copias de Variable 01)
   - {len(no_discriminan_extra)} variables sin poder discriminativo

   CONSERVAR (dos estrategias posibles):
   A) Agresiva: solo representante de cada grupo correlado
      → ~32 features independientes efectivas
   B) Conservadora: todas las que discriminan (p<0.05)  
      → 68 features (dejamos que el modelo decida)

   RECOMENDACIÓN: empezar con B y dejar que 
   LightGBM + feature importance haga la selección final
""")

=== RESUMEN FINAL DE EXPLORACIÓN ===

1. REDUNDANTES EXACTAS (copias de Variable 01): 9
   ['Variable 24', 'Variable 25', 'Variable 26', 'Variable 36', 'Variable 37', 'Variable 38', 'Variable 48', 'Variable 49', 'Variable 50']

2. NO DISCRIMINAN (p>=0.05, sin contar redundantes): 54

3. DISCRIMINAN (p<0.05): 68
   Muy significativas (p<0.001): 39

4. GRUPOS DE FEATURES CORRELADAS (>0.8):
   9 grupos → 44 variables colapsables a 9 representantes
   Representantes: Variable 29, 07, 47, 10, 42, 53, 63, 74, 82

=== DECISIONES PARA EL DATASET LIMPIO ===

   ELIMINAR:
   - 9 variables redundantes exactas (copias de Variable 01)
   - 54 variables sin poder discriminativo

   CONSERVAR (dos estrategias posibles):
   A) Agresiva: solo representante de cada grupo correlado
      → ~32 features independientes efectivas
   B) Conservadora: todas las que discriminan (p<0.05)  
      → 68 features (dejamos que el modelo decida)

   RECOMENDACIÓN: empezar con B y dejar que 
   LightGBM + feature impo

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import lightgbm as lgb

# ── 1. DATASET LIMPIO ─────────────────────────────────────────────────────────
features_eliminar = redundantes + no_discriminan_extra
features_mantener = [c for c in basic.columns 
                     if c != 'Variable de Salida' 
                     and c not in features_eliminar]

print(f"Features originales:  {basic.shape[1] - 1}")
print(f"Features eliminadas:  {len(features_eliminar)}")
print(f"Features finales:     {len(features_mantener)}")

X = basic[features_mantener]
y = basic['Variable de Salida']

Features originales:  131
Features eliminadas:  63
Features finales:     68


In [22]:
# ── 2. SPLIT ESTRATIFICADO ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"\nTrain: {X_train.shape} | Test: {X_test.shape}")
print(f"Distribución train — NOK: {y_train.mean():.3f} | OK: {1-y_train.mean():.3f}")



Train: (7996, 68) | Test: (1999, 68)
Distribución train — NOK: 0.756 | OK: 0.244


In [23]:
# ── 3. MODELO BASE (sin tunear, para tener referencia) ────────────────────────
model_base = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    is_unbalance=True,
    random_state=SEED,
    verbose=-1
)
model_base.fit(X_train, y_train)
y_pred_base = model_base.predict(X_test)

print(f"\n=== MODELO BASE — LightGBM sin tunear ===")
print(f"F1 macro: {f1_score(y_test, y_pred_base, average='macro'):.4f}")
print(confusion_matrix(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base))


=== MODELO BASE — LightGBM sin tunear ===
F1 macro: 0.5442
[[ 148  340]
 [ 326 1185]]
              precision    recall  f1-score   support

           0       0.31      0.30      0.31       488
           1       0.78      0.78      0.78      1511

    accuracy                           0.67      1999
   macro avg       0.54      0.54      0.54      1999
weighted avg       0.66      0.67      0.67      1999



In [24]:
import pandas as pd
import matplotlib.pyplot as plt

# Feature importance
importance = pd.DataFrame({
    'feature': features_mantener,
    'importance': model_base.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("=== TOP 30 FEATURES MÁS IMPORTANTES ===")
print(importance.head(30).to_string(index=False))

print(f"\n=== FEATURES CON IMPORTANCIA CERO ===")
cero = importance[importance['importance'] == 0]['feature'].tolist()
print(f"{len(cero)} features ignoradas por el modelo:")
print(cero)

=== TOP 30 FEATURES MÁS IMPORTANTES ===
     feature  importance
 Variable 58         591
 Variable 10         526
 Variable 99         459
 Variable 94         430
 Variable 95         421
 Variable 82         419
 Variable 62         407
 Variable 98         389
 Variable 44         382
 Variable 14         368
 Variable 74         367
 Variable 23         349
 Variable 42         335
 Variable 28         334
 Variable 32         331
 Variable 31         316
 Variable 43         315
Variable 100         303
 Variable 29         293
 Variable 47         283
 Variable 27         278
 Variable 83         265
 Variable 69         259
 Variable 19         256
 Variable 75         254
 Variable 30         253
 Variable 16         252
 Variable 11         248
 Variable 52         243
 Variable 51         234

=== FEATURES CON IMPORTANCIA CERO ===
1 features ignoradas por el modelo:
['Variable 101_CA']


In [26]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 1500),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 200),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight':  trial.suggest_float('scale_pos_weight', 1.0, 6.0),
        'random_state':      SEED,
        'verbose':           -1,
    }
    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    score = cross_val_score(model, X_train, y_train,
                            cv=cv, scoring='f1_macro', n_jobs=-1).mean()
    return score

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\n=== RESULTADO OPTUNA ===")
print(f"Mejor F1 macro (CV): {study.best_value:.4f}")
print(f"Mejores parámetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 52. Best value: 0.534086: 100%|██████████| 100/100 [35:00<00:00, 21.00s/it]


=== RESULTADO OPTUNA ===
Mejor F1 macro (CV): 0.5341
Mejores parámetros:
  n_estimators: 1340
  learning_rate: 0.19609213134917472
  num_leaves: 162
  max_depth: 12
  min_child_samples: 21
  subsample: 0.9402991124931198
  colsample_bytree: 0.5231403754140721
  reg_alpha: 1.204234968888754e-07
  reg_lambda: 2.990397281642616e-05
  scale_pos_weight: 1.239991405341637


In [27]:
# Modelo con mejores parámetros de Optuna
model_tuned = lgb.LGBMClassifier(**study.best_params, random_state=SEED, verbose=-1)
model_tuned.fit(X_train, y_train)
y_pred_tuned = model_tuned.predict(X_test)

print(f"=== MODELO TUNEADO ===")
print(f"F1 macro: {f1_score(y_test, y_pred_tuned, average='macro'):.4f}")
print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))

# Comparar con búsqueda de umbral óptimo
probs = model_tuned.predict_proba(X_test)[:, 0]  # probabilidad de clase OK
thresholds = np.arange(0.1, 0.9, 0.01)
best_t, best_f1 = 0.5, 0
for t in thresholds:
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"\n=== CON UMBRAL AJUSTADO ===")
print(f"Umbral óptimo: {best_t:.2f}")
print(f"F1 macro:      {best_f1:.4f}")
y_pred_threshold = (probs >= best_t).astype(int)
print(confusion_matrix(y_test, y_pred_threshold))
print(classification_report(y_test, y_pred_threshold))

=== MODELO TUNEADO ===
F1 macro: 0.5314
[[  82  406]
 [ 143 1368]]
              precision    recall  f1-score   support

           0       0.36      0.17      0.23       488
           1       0.77      0.91      0.83      1511

    accuracy                           0.73      1999
   macro avg       0.57      0.54      0.53      1999
weighted avg       0.67      0.73      0.69      1999


=== CON UMBRAL AJUSTADO ===
Umbral óptimo: 0.10
F1 macro:      0.3002
[[ 374  114]
 [1278  233]]
              precision    recall  f1-score   support

           0       0.23      0.77      0.35       488
           1       0.67      0.15      0.25      1511

    accuracy                           0.30      1999
   macro avg       0.45      0.46      0.30      1999
weighted avg       0.56      0.30      0.27      1999



In [28]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

# Base learners diversos
estimators = [
    ('lgbm', lgb.LGBMClassifier(
        **study.best_params, random_state=SEED, verbose=-1)),
    ('xgb', xgb.XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        scale_pos_weight=3, random_state=SEED, verbosity=0)),
    ('rf', RandomForestClassifier(
        n_estimators=300, max_depth=10, class_weight='balanced',
        random_state=SEED, n_jobs=-1)),
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=0.1, class_weight='balanced'),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    passthrough=False,
    n_jobs=-1
)

stack.fit(X_train, y_train)
y_pred_stack = stack.predict(X_test)

print("=== STACKING ===")
print(f"F1 macro: {f1_score(y_test, y_pred_stack, average='macro'):.4f}")
print(confusion_matrix(y_test, y_pred_stack))
print(classification_report(y_test, y_pred_stack))

=== STACKING ===
F1 macro: 0.5577
[[291 197]
 [603 908]]
              precision    recall  f1-score   support

           0       0.33      0.60      0.42       488
           1       0.82      0.60      0.69      1511

    accuracy                           0.60      1999
   macro avg       0.57      0.60      0.56      1999
weighted avg       0.70      0.60      0.63      1999

